### Imports

In [1]:
import sys
sys.dont_write_bytecode = True


import torch
import numpy as np
import random
import os

def set_seeds(seed_value=42):
    """Sets seeds for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)

set_seeds(42) 

import json
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import warnings
import logging
from datetime import datetime
warnings.filterwarnings('ignore')

from model import get_model
from config import CFG
from dataset import get_dataset_class
from transform import get_transforms
from runner import run_baseline, run_lodo

torch.manual_seed(CFG["system"]["seed"])
np.random.seed(CFG["system"]["seed"])

device = CFG["system"]["device"]
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

DS = "PACS"
MODEL_NAME = "resnet50"

Device: cuda
PyTorch: 2.8.0+cu126


### DataLoading

In [2]:
train_transform, test_transform = get_transforms(img_size=224, augment=False, use_imagenet_norm=False)

DatasetClass = get_dataset_class(DS)

ld = DatasetClass(
    data_root=CFG["datasets"][DS]["root"],
    transform=train_transform,
    batch_size=CFG["train"]["batch_size"]
)

print("\nData loaders ready!")


Data loaders ready!


### Logging

In [3]:
dataset_name = DS
base_dir = os.path.join(os.getcwd(), dataset_name)
subdirs = ["logs", "checkpoints", "plots"]

for sub in subdirs:
    os.makedirs(os.path.join(base_dir, sub), exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = os.path.join(base_dir, "logs", f"train_{timestamp}.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(f"{dataset_name}_logger")

logger.info(f"Initialized experiment directories for {dataset_name}")
logger.info(f"Logs: {os.path.join(base_dir, 'logs')}")
logger.info(f"Checkpoints: {os.path.join(base_dir, 'checkpoints')}")
logger.info(f"Plots: {os.path.join(base_dir, 'plots')}")

2025-12-01 11:20:28,864 | INFO | Initialized experiment directories for PACS
2025-12-01 11:20:28,864 | INFO | Logs: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet50_experiments\PACS\logs
2025-12-01 11:20:28,864 | INFO | Checkpoints: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet50_experiments\PACS\checkpoints
2025-12-01 11:20:28,864 | INFO | Plots: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet50_experiments\PACS\plots


### Setup

In [4]:
domains = CFG["datasets"][DS]["domains"]
loaders = {d: {"train": ld.get_dataloader(d, train=True), "val": ld.get_dataloader(d, train=False)} for d in domains}
ckpt_root = os.path.join(base_dir, "checkpoints")
log_dir = os.path.join(base_dir, "logs")
plots_dir = os.path.join(base_dir, "plots")
os.makedirs(ckpt_root, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
model_factory = lambda cfg, dataset_key: get_model(cfg,dataset=DS)
optimizer_fn = lambda model: optim.AdamW(model.parameters(), lr=CFG["train"]["lr"], weight_decay=CFG["train"].get("weight_decay", 0.01))
device = CFG["system"]["device"]
epochs = CFG["train"]["epochs"]

{
  "lodo_results": {
    "art_painting": 0.8341463414634146,
    "cartoon": 0.7974413646055437,
    "photo": 0.9580838323353293,
    "sketch": 0.6017811704834606
  },
  "timestamp": "20251004_020611"
}

### Leave One Domain Out

In [ ]:
lodo_results, lodo_mean, lodo_summary = run_lodo(
    model_fn=model_factory,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    ckpt_root=ckpt_root,
    log_dir=log_dir,
    epochs=epochs
)

2025-12-01 11:20:29,813 | INFO | === LODO: Leaving out domain 'art_painting' ===



=== LODO: Leaving out domain 'art_painting' ===


2025-12-01 11:20:58,251 | INFO | [art_painting] Epoch 1/10 | Train - Loss: 0.5274, Cls: 0.5086, GRQO: 0.0189, Acc: 0.8270 | Val - Loss: 0.7056, Cls: 0.7029, GRQO: 0.0026, Acc: 0.7822


[art_painting] Epoch 1/10 | Train - Loss: 0.5274, Cls: 0.5086, GRQO: 0.0189, Acc: 0.8270 | Val - Loss: 0.7056, Cls: 0.7029, GRQO: 0.0026, Acc: 0.7822


2025-12-01 11:20:58,480 | INFO | [art_painting] New best val acc: 0.7822


...

[cartoon] Epoch 2/10 | Train - Loss: 0.0931, Cls: 0.0917, GRQO: 0.0014, Acc: 0.9693 | Val - Loss: 0.7423, Cls: 0.7401, GRQO: 0.0022, Acc: 0.7905


2025-12-01 11:26:12,291 | INFO | [cartoon] New best val acc: 0.7905


[cartoon] New best val acc: 0.7905


2025-12-01 11:26:40,038 | INFO | [cartoon] Epoch 3/10 | Train - Loss: 0.0503, Cls: 0.0495, GRQO: 0.0007, Acc: 0.9850 | Val - Loss: 1.2899, Cls: 1.2870, GRQO: 0.0029, Acc: 0.6937


[cartoon] Epoch 3/10 | Train - Loss: 0.0503, Cls: 0.0495, GRQO: 0.0007, Acc: 0.9850 | Val - Loss: 1.2899, Cls: 1.2870, GRQO: 0.0029, Acc: 0.6937


### Baseline

In [5]:
baseline_results, baseline_mean = run_baseline(
    model_name=MODEL_NAME,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    epochs=CFG["train"]["epochs"]
)

2025-12-01 10:58:58,142 | INFO | Initializing ResNet baseline: resnet50


Initializing ResNet baseline: resnet50


2025-12-01 10:58:58,422 | INFO | === Baseline LODO: Leaving out domain 'art_painting' ===



=== Baseline LODO: Leaving out domain 'art_painting' ===


2025-12-01 10:59:22,144 | INFO | [art_painting] Epoch 1/10 | Train - Loss: 0.4189, Acc: 0.8740 | Val Acc: 0.8022


...

[sketch] Epoch 8/10 | Train - Loss: 0.0007, Acc: 1.0000 | Val Acc: 0.7160


2025-12-01 11:14:09,152 | INFO | [sketch] Epoch 9/10 | Train - Loss: 0.0007, Acc: 0.9998 | Val Acc: 0.7213


[sketch] Epoch 9/10 | Train - Loss: 0.0007, Acc: 0.9998 | Val Acc: 0.7213


2025-12-01 11:14:32,485 | INFO | [sketch] Epoch 10/10 | Train - Loss: 0.0018, Acc: 0.9995 | Val Acc: 0.6396
2025-12-01 11:14:32,485 | INFO | [sketch] Best Val Acc: 0.7223
2025-12-01 11:14:32,485 | INFO | ------------------------------------------------------------
2025-12-01 11:14:32,485 | INFO | Baseline LODO (resnet50) finished | Mean Acc: 0.8384


[sketch] Epoch 10/10 | Train - Loss: 0.0018, Acc: 0.9995 | Val Acc: 0.6396
[sketch] Best Val Acc: 0.7223
------------------------------------------------------------
Baseline LODO (resnet50) finished | Mean Acc: 0.8384


In [ ]:
"grqo": {
        "alpha": 0.5,
        "beta": 0.5,
        "tau": 3e-3,
        "temperature": 0.3,
        "hidden_dim": 192,
        "num_heads": 4,
        "num_tokens": 32,
        "num_layers":4,
        "dropout": 0.1,
        "ddropout": 0.1,
        "lambda_grqo": 0.7,
        "teacher_ema": 0.95,
        "reward_proxy": "taylor",  # or "gradnorm"
        "random_k":None
    },